# 04_01_proportion_analysis

This notebook conducts the one-sample proportion analysis for the behavior variable **SadOrHopeless**.

Research question:

**Is the proportion of students with `SadOrHopeless = 1` different from the benchmark value 0.30?**

## Step 1: Import packages

In [1]:
import os
import math
import numpy as np
import pandas as pd
from scipy.stats import norm

## Step 2: Load processed data
Read the processed dataset created earlier.

In [2]:
file_path = "../data/processed/cycle2_vddc_imputed.csv"
df = pd.read_csv(file_path)

df.head()

,SadOrHopeless,HowMuchDoYouWeighWithoutShoesInKG
0,1,50.80
1,2,68.04
2,1,52.16
3,1,79.38
4,1,60.78


## Step 3: Confirm analysis variable and define success
- Behavior variable: `SadOrHopeless`
- Success = code `1`
- Failure = code `2`
- Benchmark proportion: `p0 = 0.30`

In [3]:
behavior_var = "SadOrHopeless"
success_code = 1
failure_code = 2
p0 = 0.30

analysis_df = df[[behavior_var]].copy()
analysis_df = analysis_df[analysis_df[behavior_var].isin([success_code, failure_code])].copy()

analysis_df.head()

,SadOrHopeless
0,1
1,2
2,1
3,1
4,1


## Step 4: Compute the sample proportion
Compute:
- sample size `n`
- number of successes
- sample proportion `p̂`

In [4]:
n = len(analysis_df)
success_count = int((analysis_df[behavior_var] == success_code).sum())
failure_count = int((analysis_df[behavior_var] == failure_code).sum())
p_hat = success_count / n

sample_summary = pd.DataFrame({
    "measure": ["sample_size_n", "success_count", "failure_count", "sample_proportion_p_hat"],
    "value": [n, success_count, failure_count, p_hat]
})

sample_summary

,measure,value
0,sample_size_n,14041.00000
1,success_count,4205.00000
2,failure_count,9836.00000
3,sample_proportion_p_hat,0.29948


**Observation:** This table shows the sample size, the number of students classified as success, and the sample proportion used for the one-sample proportion inference.

## Step 5: State the benchmark proportion
The benchmark value for this analysis is:

- `p0 = 0.30`

In [5]:
benchmark_table = pd.DataFrame({
    "parameter": ["population_proportion_p"],
    "benchmark_p0": [p0]
})

benchmark_table

,parameter,benchmark_p0
0,population_proportion_p,0.3


**Observation:** The sample proportion will be compared with the benchmark proportion 0.30 in both the confidence interval and the hypothesis test.

## Step 6: Construct a 95% confidence interval for the population proportion
Use the normal-approximation confidence interval:

`p̂ ± z_(0.025) * sqrt(p̂(1-p̂)/n)`

where `z_(0.025) = 1.96` for a 95% confidence level.

In [6]:
confidence_level = 0.95
alpha = 1 - confidence_level
z_critical = norm.ppf(1 - alpha / 2)

se_ci = math.sqrt(p_hat * (1 - p_hat) / n)
margin_error = z_critical * se_ci
ci_lower = p_hat - margin_error
ci_upper = p_hat + margin_error

ci_table = pd.DataFrame({
    "confidence_level": [confidence_level],
    "p_hat": [p_hat],
    "standard_error": [se_ci],
    "z_critical": [z_critical],
    "margin_of_error": [margin_error],
    "ci_lower": [ci_lower],
    "ci_upper": [ci_upper]
})

ci_table

,confidence_level,p_hat,standard_error,z_critical,margin_of_error,ci_lower,ci_upper
0,0.95,0.29948,0.003865,1.959964,0.007576,0.291904,0.307056


**Observation:** The confidence interval gives a plausible range for the population proportion of students classified as `SadOrHopeless = 1`.

## Step 7: Conduct a one-sample test for the population proportion
Hypotheses:

- `H0: p = 0.30`
- `H1: p ≠ 0.30`

Use the one-sample z test for a population proportion.

In [7]:
se_test = math.sqrt(p0 * (1 - p0) / n)
z_stat = (p_hat - p0) / se_test
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

decision = "Reject H0" if p_value < alpha else "Do not reject H0"

test_table = pd.DataFrame({
    "null_hypothesis": ["p = 0.30"],
    "alternative_hypothesis": ["p != 0.30"],
    "sample_proportion_p_hat": [p_hat],
    "benchmark_p0": [p0],
    "standard_error_under_H0": [se_test],
    "z_statistic": [z_stat],
    "p_value": [p_value],
    "alpha": [alpha],
    "decision": [decision]
})

test_table

,null_hypothesis,alternative_hypothesis,sample_proportion_p_hat,benchmark_p0,standard_error_under_H0,z_statistic,p_value,alpha,decision
0,p = 0.30,p != 0.30,0.29948,0.3,0.003867,-0.134436,0.893058,0.05,Do not reject H0


**Observation:** The p-value and decision show whether the observed sample proportion is statistically different from the benchmark value 0.30.

## Step 8: Save the main inferential results
Save the main results to `../outputs/tables/`.

In [8]:
os.makedirs("../outputs/tables", exist_ok=True)

sample_summary.to_csv("../outputs/tables/proportion_sample_summary.csv", index=False)
benchmark_table.to_csv("../outputs/tables/proportion_benchmark_table.csv", index=False)
ci_table.to_csv("../outputs/tables/proportion_confidence_interval.csv", index=False)
test_table.to_csv("../outputs/tables/proportion_hypothesis_test.csv", index=False)

summary_results = pd.DataFrame({
    "analysis": ["Proportion Analysis"],
    "variable": [behavior_var],
    "sample_size_n": [n],
    "sample_proportion_p_hat": [p_hat],
    "benchmark_p0": [p0],
    "confidence_level": [confidence_level],
    "ci_lower": [ci_lower],
    "ci_upper": [ci_upper],
    "z_statistic": [z_stat],
    "p_value": [p_value],
    "decision": [decision]
})

summary_results.to_csv("../outputs/tables/proportion_inference_summary.csv", index=False)

summary_results

,analysis,variable,sample_size_n,sample_proportion_p_hat,benchmark_p0,confidence_level,ci_lower,ci_upper,z_statistic,p_value,decision
0,Proportion Analysis,SadOrHopeless,14041,0.29948,0.3,0.95,0.291904,0.307056,-0.134436,0.893058,Do not reject H0


**Observation:** This summary table collects the main inferential results for the one-sample proportion analysis in one place.

## Step 9: Interpret the results in context

In [9]:
if p_value < alpha:
    interpretation_text = (
        f"For SadOrHopeless, the sample proportion is {p_hat:.4f}. "
        f"The 95% confidence interval is ({ci_lower:.4f}, {ci_upper:.4f}). "
        f"Because the p-value is {p_value:.4f}, which is smaller than {alpha:.2f}, "
        f"we reject H0 and conclude that the population proportion of students with "
        f"SadOrHopeless = 1 is significantly different from 0.30."
    )
else:
    interpretation_text = (
        f"For SadOrHopeless, the sample proportion is {p_hat:.4f}. "
        f"The 95% confidence interval is ({ci_lower:.4f}, {ci_upper:.4f}). "
        f"Because the p-value is {p_value:.4f}, which is greater than or equal to {alpha:.2f}, "
        f"we do not reject H0. The data do not provide sufficient evidence that the population "
        f"proportion of students with SadOrHopeless = 1 is different from 0.30."
    )

print(interpretation_text)

For SadOrHopeless, the sample proportion is 0.2995. The 95% confidence interval is (0.2919, 0.3071). Because the p-value is 0.8931, which is greater than or equal to 0.05, we do not reject H0. The data do not provide sufficient evidence that the population proportion of students with SadOrHopeless = 1 is different from 0.30.
